# Model setup + Lasso & XGBoost models

Shared train/test split, rolling-origin CV folds, and evaluation metrics,
now combined with two working models (Lasso and XGBoost) so the whole
pipeline runs end-to-end in one notebook.

**Fixes applied vs. the original shared setup file:**
- `FEATURE_COLS` now excludes raw balance-sheet levels (`toas`, `ncli`,
  `culi`, `shfd`, ...). Those are collinear with each other (accounting
  identities: `toas == tshf`, `ncli == ltdb + oncl`, etc.) and contain
  extreme outliers (`toas` up to ~7.9e14 in the real data) — feeding them
  into unregularized/linear models makes coefficients unstable. The
  engineered ratios (`leverage`, `solvency`, ...) and `log_toas`/`log_empl`
  are used instead.
- `train_df` / `test_df` are explicitly `.copy()`'d right after the split
  to silence `SettingWithCopyWarning` and avoid mutating a view.
- Numeric columns are downcast to `float32` and each fold loop explicitly
  frees fitted objects (`del` + `gc.collect()`) to reduce peak memory —
  this is what caused the `MemoryError` in the original notebook (it died
  on fold 3, ~520K rows, after folds 1–2 succeeded).

## 1. Load data

In [ ]:
import gc
import numpy as np
import pandas as pd

DATA_PATH = "data/model_input/analysis_panel.parquet"  # <- adjust to your actual path

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(30000, 77)


,report_date,idnr,name,type,dateinc,naics_core_code,closdate_year,empl,ncliGrowthNextYear,fias,...,ifo_business_climate_growth,ifo_business_situation_growth,ifo_business_expectations_growth,de_economic_sentiment_index_growth,de_employment_expectations_index_growth,de_industry_confidence_growth,de_services_confidence_growth,de_consumer_confidence_growth,de_retail_confidence_growth,de_construction_confidence_growth
0,2022-12-01,DE0000000,FIRM_0,Limited liability company - GmbH,1990-01-01,3849,2022,NaN,0.072237,63.873200,...,-0.927779,-0.367576,1.920362,-0.145454,-0.589240,-23.729220,46.904355,88.288271,87.759264,10.991356
1,2015-12-01,DE0000001,FIRM_1,Limited partnership - KG,1990-01-01,1226,2015,116.0,-0.148518,444.681822,...,-2.320115,1.594225,-2.836448,-0.257534,-0.585246,244.615467,-69.109572,36.712725,105.157748,-56.469359
2,2010-12-01,DE0000002,FIRM_2,Public limited company - AG,1990-01-01,7951,2010,NaN,0.096697,104.902058,...,0.561252,-1.394983,-1.007956,-0.377132,0.673660,13.764739,-20.203989,inf,103.207428,75.405151
3,2013-12-01,DE0000003,FIRM_3,Public limited company - AG,1990-01-01,7753,2013,484.0,0.129630,276.661337,...,2.736458,0.661454,-0.052987,-2.727005,-1.802926,-11.418988,-195.943129,124.486620,104.313351,-150.295442
4,2021-12-01,DE0000004,FIRM_4,Public limited company - AG,2001-05-01,8983,2021,384.0,-0.625989,13.675424,...,1.719144,-0.776004,-0.462358,-0.256972,-1.180845,-141.859605,-128.348487,91.635711,-41.765002,-30.593854


In [ ]:
df['report_year'] = pd.to_datetime(df['report_date']).dt.year
df = df.drop(columns=['report_date'])
df.head(3)

,idnr,name,type,dateinc,naics_core_code,closdate_year,empl,ncliGrowthNextYear,fias,ifas,...,ifo_business_situation_growth,ifo_business_expectations_growth,de_economic_sentiment_index_growth,de_employment_expectations_index_growth,de_industry_confidence_growth,de_services_confidence_growth,de_consumer_confidence_growth,de_retail_confidence_growth,de_construction_confidence_growth,report_year
0,DE0000000,FIRM_0,Limited liability company - GmbH,1990-01-01,3849,2022,NaN,0.072237,63.873200,360.568660,...,-0.367576,1.920362,-0.145454,-0.589240,-23.729220,46.904355,88.288271,87.759264,10.991356,2022
1,DE0000001,FIRM_1,Limited partnership - KG,1990-01-01,1226,2015,116.0,-0.148518,444.681822,11662.387887,...,1.594225,-2.836448,-0.257534,-0.585246,244.615467,-69.109572,36.712725,105.157748,-56.469359,2015
2,DE0000002,FIRM_2,Public limited company - AG,1990-01-01,7951,2010,NaN,0.096697,104.902058,1.452407,...,-1.394983,-1.007956,-0.377132,0.673660,13.764739,-20.203989,inf,103.207428,75.405151,2010


## 2. Config

`FEATURE_COLS` is built by exclusion, so it automatically stays in sync if
new engineered columns get added upstream — but the exclusion list is now
explicit about *why* each group is dropped.

In [ ]:
YEAR_COL = "report_year"
TARGET   = "ncliGrowthNextYear"

# Identifiers / redundant-year info -> never used as features
ID_COLS = ['idnr', 'name', 'dateinc', 'naics_core_code', 'closdate_year']

# Raw balance-sheet levels: EXCLUDED as features.
# - Collinear with each other and with the engineered ratios built from them
#   (toas == tshf; ncli == ltdb + oncl; cuas == stok+debt+ocas+cash; etc.)
# - Contain extreme outliers (toas up to ~7.9e14 in the real data) that
#   destabilize unregularized/linear models.
# Use the engineered ratios / log_toas / log_empl instead.
RAW_LEVEL_COLS = [
    'fias', 'ifas', 'tfas', 'ofas', 'cuas', 'stok', 'debt', 'ocas', 'cash',
    'toas', 'shfd', 'capi', 'osfd', 'ncli', 'ltdb', 'oncl', 'prov', 'culi',
    'loan', 'cred', 'ocli', 'tshf', 'wkca',
]

EXCLUDE_COLS = ID_COLS + RAW_LEVEL_COLS

FEATURE_COLS = df.columns.difference([YEAR_COL, TARGET, *EXCLUDE_COLS]).tolist()
CATEGORY_COLS = ['naics_2digit', 'type']  # one-hot encoded

print(f"{len(FEATURE_COLS)} feature columns (engineered ratios + macro/sentiment + categoricals):")
print(FEATURE_COLS)

MIN_TRAIN_YEARS = 5
TEST_YEARS = [2020, 2021, 2022, 2023]

47 feature columns (engineered ratios + macro/sentiment + categoricals):
['cash_growth', 'cash_ratio', 'constraining_ratio', 'current_ratio', 'de_construction_confidence', 'de_construction_confidence_growth', 'de_consumer_confidence', 'de_consumer_confidence_growth', 'de_economic_sentiment_index', 'de_economic_sentiment_index_growth', 'de_employment_expectations_index', 'de_employment_expectations_index_growth', 'de_industry_confidence', 'de_industry_confidence_growth', 'de_retail_confidence', 'de_retail_confidence_growth', 'de_services_confidence', 'de_services_confidence_growth', 'empl', 'firm_age', 'gearing', 'growth_volatility', 'ifo_business_climate', 'ifo_business_climate_growth', 'ifo_business_expectations', 'ifo_business_expectations_growth', 'ifo_business_situation', 'ifo_business_situation_growth', 'inventory_share', 'is_na_empl', 'leverage', 'litigious_ratio', 'lm_polarity', 'log_empl', 'log_toas', 'naics_2digit', 'ncliGrowthThisYear', 'quick_ratio', 'receivables_share', 'so

Reasoning for `MIN_TRAIN_YEARS = 5`:
- Early years have far fewer, differently-composed firms (panel coverage
  expanded over time).
- If a CV fold trains only on those sparse years, its score reflects a
  coverage-composition shift, not real forecasting difficulty.
- `MIN_TRAIN_YEARS = 5` dilutes the sparse years to roughly ~15-20% of
  fold 1's training set (see the table below).

In [ ]:
year_counts = df[YEAR_COL].value_counts().sort_index()
print(year_counts)

years_sorted = sorted(year_counts.index)
SPARSE_CUTOFF_YEAR = 2013  # adjust to wherever coverage visibly stabilizes above

print(f"\n{'min_train_years':>16} {'first_test_year':>16} {'sparse_share':>14}")
for m in range(4, 10):
    if m >= len(years_sorted):
        continue
    train_years = years_sorted[:m]
    total = sum(year_counts[y] for y in train_years)
    sparse = sum(year_counts[y] for y in train_years if y < SPARSE_CUTOFF_YEAR)
    share = sparse / total if total else float("nan")
    print(f"{m:>16} {years_sorted[m]:>16} {share:>13.1%}")

report_year
2010    2134
2011    2147
2012    2130
2013    2163
2014    2107
2015    2176
2016    2094
2017    2093
2018    2128
2019    2158
2020    2142
2021    2196
2022    2189
2023    2143
Name: count, dtype: int64

 min_train_years  first_test_year   sparse_share
               4             2014         74.8%
               5             2015         60.0%
               6             2016         49.9%
               7             2017         42.9%
               8             2018         37.6%
               9             2019         33.4%


**Why the test set starts at 2020 (isolating COVID on the test side only):**
this measures "if an unprecedented shock hits and the model has never seen
anything like it, how badly does it break?" — the realistic deployment
scenario for a bank planning financing programmes, which can't guarantee
the next crisis looks like anything in its training data.

## 3. Fold + metric functions

In [ ]:
def rolling_origin_folds(years, min_train_years=4):
    """Expanding-window folds: train on years[:i], test on years[i].

    Example with years 2010..2019, min_train_years=5:
        Fold 1: train <=2014, test 2015
        Fold 2: train <=2015, test 2016
        ...
        Fold 5: train <=2018, test 2019
    """
    years = sorted(set(years))
    return [(years[:i], years[i]) for i in range(min_train_years, len(years))]


def regression_metrics(y_true, y_pred, baseline_value):
    """RMSE, MAE, R2_oos vs. a naive baseline (e.g. train-set mean).

    R2_oos > 0 means the model beats "always predict the baseline value".
    Use the SAME baseline_value (train-set mean) across models being compared.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    yt, yp = y_true[mask], y_pred[mask]

    if len(yt) == 0:
        return {"RMSE": np.nan, "MAE": np.nan, "R2_oos": np.nan, "n": 0}

    err = yt - yp
    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae = float(np.mean(np.abs(err)))
    ss_res = np.sum(err ** 2)
    ss_baseline = np.sum((yt - baseline_value) ** 2)
    r2_oos = float(1 - ss_res / ss_baseline) if ss_baseline > 0 else np.nan

    return {"RMSE": rmse, "MAE": mae, "R2_oos": r2_oos, "n": int(mask.sum())}

## 4. Train/test split + CV folds

- **Train/development:** 2010–2019
- **Final test:** 2020–2023 (set aside, never touched during tuning)
- **Rolling validation:** expanding-window CV folds built from the train
  set only, so test years can never leak into tuning.

In [ ]:
# Replace +/-inf (from pct_change() on variables that can be zero/negative,
# e.g. de_consumer_confidence_growth, cash_growth) BEFORE splitting, so every
# fold inherits the fix.
df[FEATURE_COLS] = df[FEATURE_COLS].replace([np.inf, -np.inf], np.nan)

# Downcast numeric feature columns to float32 to roughly halve memory use
# during imputation/one-hot encoding across CV folds.
numeric_cols_all = [c for c in FEATURE_COLS if c not in CATEGORY_COLS]
df[numeric_cols_all] = df[numeric_cols_all].astype("float32")

train_df = df[df[YEAR_COL] < min(TEST_YEARS)].copy()
test_df  = df[df[YEAR_COL].isin(TEST_YEARS)].copy()
baseline_value = train_df[TARGET].mean()  # SAME baseline for every model

print(f"train: {train_df.shape}  (years {train_df[YEAR_COL].min()}-{train_df[YEAR_COL].max()})")
print(f"test:  {test_df.shape}  (years {test_df[YEAR_COL].min()}-{test_df[YEAR_COL].max()})")
print(f"baseline_value (train mean): {baseline_value:.4f}")

train: (21330, 77)  (years 2010-2019)
test:  (8670, 77)  (years 2020-2023)
baseline_value (train mean): -0.0517


In [ ]:
years = train_df[YEAR_COL].unique()
cv_folds = rolling_origin_folds(years, min_train_years=MIN_TRAIN_YEARS)

for train_years, test_year in cv_folds:
    print(f"train <= {max(train_years)} ({len(train_years)} yrs)  ->  test {test_year}")

train <= 2014 (5 yrs)  ->  test 2015
train <= 2015 (6 yrs)  ->  test 2016
train <= 2016 (7 yrs)  ->  test 2017
train <= 2017 (8 yrs)  ->  test 2018
train <= 2018 (9 yrs)  ->  test 2019


## 5. Model 1 — Lasso (regularized linear model)

Preferred over plain OLS here: the engineered ratios are correlated with
each other by construction (e.g. `leverage` and `solvency` are
near-mirrors), so Lasso's L1 penalty gives more stable coefficients and
doubles as feature selection — useful for the "model interpretation"
part of the deliverable.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols = [c for c in FEATURE_COLS if c not in CATEGORY_COLS]

def make_lasso_pipeline():
    preprocess = ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric_cols),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), CATEGORY_COLS),
    ])
    return Pipeline([
        ("preprocess", preprocess),
        ("model", LassoCV(cv=3, max_iter=5000, n_jobs=-1, random_state=42)),
    ])

print("Tuning Lasso across CV folds...")
for train_years, test_year in cv_folds:
    fold_train = train_df[train_df[YEAR_COL].isin(train_years)]
    fold_test  = train_df[train_df[YEAR_COL] == test_year]

    pipe = make_lasso_pipeline()
    pipe.fit(fold_train[FEATURE_COLS], fold_train[TARGET])
    preds = pipe.predict(fold_test[FEATURE_COLS])

    m = regression_metrics(fold_test[TARGET].values, preds, fold_train[TARGET].mean())
    print(test_year, m)

    del pipe
    gc.collect()

Tuning Lasso across CV folds...


2015 {'RMSE': 0.30213613715217574, 'MAE': 0.24062572877549315, 'R2_oos': 0.0, 'n': 2176}


2016 {'RMSE': 0.30548112123058135, 'MAE': 0.24362724046253184, 'R2_oos': 0.0, 'n': 2094}


2017 {'RMSE': 0.290611855302066, 'MAE': 0.2330160899417062, 'R2_oos': 0.0, 'n': 2093}


2018 {'RMSE': 0.2976601407423579, 'MAE': 0.2363757498711858, 'R2_oos': 0.0, 'n': 2128}


2019 {'RMSE': 0.29612384112316403, 'MAE': 0.2341746753440294, 'R2_oos': 0.0, 'n': 2158}


In [ ]:
# Final refit on the full train set, scored on the held-out 2020-2023 test set
lasso_final = make_lasso_pipeline()
lasso_final.fit(train_df[FEATURE_COLS], train_df[TARGET])
lasso_preds = lasso_final.predict(test_df[FEATURE_COLS])

lasso_metrics = regression_metrics(test_df[TARGET].values, lasso_preds, baseline_value)
print("Lasso final (2020-2023 test):", lasso_metrics)
print("chosen alpha:", lasso_final.named_steps["model"].alpha_)

Lasso final (2020-2023 test): {'RMSE': 0.3017908711351265, 'MAE': 0.24225865783187184, 'R2_oos': 0.0, 'n': 8670}
chosen alpha: 0.003906831449618139


In [ ]:
# Interpretation: which features survived regularization
feature_names = lasso_final.named_steps["preprocess"].get_feature_names_out()
coefs = pd.Series(lasso_final.named_steps["model"].coef_, index=feature_names)
nonzero = coefs[coefs != 0]
print(f"{len(nonzero)} / {len(coefs)} features survived regularization")
nonzero.reindex(nonzero.abs().sort_values(ascending=False).index).head(15).round(4)

0 / 129 features survived regularization


Series([], dtype: float64)

## 6. Model 2 — XGBoost

Handles missing values and categorical columns natively — no imputation
or one-hot encoding needed, which also keeps this model's memory
footprint much lower than the Lasso pipeline's.

In [ ]:
import xgboost as xgb

def prep_xgb_frame(frame, category_dtypes=None):
    out = frame[FEATURE_COLS].copy()
    for c in CATEGORY_COLS:
        if category_dtypes is not None:
            out[c] = out[c].astype("category").cat.set_categories(category_dtypes[c])
        else:
            out[c] = out[c].astype("category")
    return out

def make_xgb_model():
    return xgb.XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        enable_categorical=True,
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )

print("Tuning XGBoost across CV folds...")
for train_years, test_year in cv_folds:
    fold_train = train_df[train_df[YEAR_COL].isin(train_years)]
    fold_test  = train_df[train_df[YEAR_COL] == test_year]

    X_fold_train = prep_xgb_frame(fold_train)
    cat_dtypes = {c: X_fold_train[c].cat.categories for c in CATEGORY_COLS}
    X_fold_test = prep_xgb_frame(fold_test, category_dtypes=cat_dtypes)

    model = make_xgb_model()
    model.fit(X_fold_train, fold_train[TARGET])
    preds = model.predict(X_fold_test)

    m = regression_metrics(fold_test[TARGET].values, preds, fold_train[TARGET].mean())
    print(test_year, m)

    del model, X_fold_train, X_fold_test
    gc.collect()

Tuning XGBoost across CV folds...


2015 {'RMSE': 0.3098844829720758, 'MAE': 0.24707452604821037, 'R2_oos': -0.051948104495298075, 'n': 2176}


2016 {'RMSE': 0.31288936643155413, 'MAE': 0.24909127264181818, 'R2_oos': -0.04909026208324008, 'n': 2094}


2017 {'RMSE': 0.2978696473783959, 'MAE': 0.23837060783615388, 'R2_oos': -0.0505720646716763, 'n': 2093}


2018 {'RMSE': 0.3040090822406801, 'MAE': 0.2403270513425878, 'R2_oos': -0.04311394431993554, 'n': 2128}


2019 {'RMSE': 0.30351169652973464, 'MAE': 0.24053333415388303, 'R2_oos': -0.05051949518963417, 'n': 2158}


In [ ]:
# Final refit on the full train set, scored on the held-out 2020-2023 test set
X_train_xgb = prep_xgb_frame(train_df)
cat_dtypes = {c: X_train_xgb[c].cat.categories for c in CATEGORY_COLS}
X_test_xgb = prep_xgb_frame(test_df, category_dtypes=cat_dtypes)

xgb_final = make_xgb_model()
xgb_final.fit(X_train_xgb, train_df[TARGET])
xgb_preds = xgb_final.predict(X_test_xgb)

xgb_metrics = regression_metrics(test_df[TARGET].values, xgb_preds, baseline_value)
print("XGBoost final (2020-2023 test):", xgb_metrics)

XGBoost final (2020-2023 test): {'RMSE': 0.30589369433073055, 'MAE': 0.24573828069107817, 'R2_oos': -0.02737466484427875, 'n': 8670}


In [ ]:
importances = pd.Series(
    xgb_final.feature_importances_,
    index=xgb_final.get_booster().feature_names,
).sort_values(ascending=False)
importances.head(15).round(4)

naics_2digit                 0.0345
litigious_ratio              0.0237
years_in_panel               0.0236
leverage                     0.0231
working_capital_ratio        0.0229
toas_growth                  0.0228
quick_ratio                  0.0227
solvency                     0.0226
gearing                      0.0225
uncertainty_ratio            0.0224
weak_modal_ratio             0.0222
type                         0.0222
de_retail_confidence         0.0221
ifo_business_situation       0.0221
ifo_business_expectations    0.0220
dtype: float32

## 7. Final model comparison (2020–2023 held-out test set)

In [ ]:
naive_metrics = regression_metrics(
    test_df[TARGET].values,
    np.full(len(test_df), baseline_value),
    baseline_value,
)

comparison = pd.DataFrame({
    "Naive (train mean)": naive_metrics,
    "Lasso": lasso_metrics,
    "XGBoost": xgb_metrics,
}).T
comparison.round(4)

,RMSE,MAE,R2_oos,n
Naive (train mean),0.3018,0.2423,0.0000,8670.0
Lasso,0.3018,0.2423,0.0000,8670.0
XGBoost,0.3059,0.2457,-0.0274,8670.0


**Next steps / open items:**
- If Lasso or XGBoost meaningfully beat the naive baseline (`R2_oos > 0`),
  that's the headline result for the "expected out-of-sample performance"
  deliverable section.
- Consider adding a third model (e.g. Random Forest, as in the professor's
  slides) for a fuller comparison.
- The rolling-origin CV metrics per fold (Section 5/6 loops) are worth
  plotting — do R2_oos trend up/down as more training years become
  available? That's relevant to your research question about how
  predictive value evolves.